# BÁO CÁO KỸ THUẬT: THU THẬP DỮ LIỆU BÁO ĐIỆN TỬ VNEXPRESS (CUSTOM CRAWLER)
**Mục tiêu:** Thu thập bài viết tin tức tự động theo kiến trúc 3 pha: **Fetch -> Refine -> Validate**.

---

### Quy trình xử lý (3 Pha):
1. **Fetch:** Gửi HTTP request tải nội dung HTML/XML từ RSS và các trang chuyên mục.
2. **Refine:** Dùng BeautifulSoup và biểu thức chính quy (Regex) bóc tách các trường: tiêu đề, tóm tắt, nội dung, tác giả, ngày đăng, chuyên mục.
3. **Validate:** Kiểm tra tính hợp lệ (bỏ bài rỗng, bài video không có text, bài quiz dưới 50 từ) trước khi lưu vào CSV.

## 1. Khai báo Thư viện & Thiết lập Cấu hình

In [1]:
import os
import re
import sys
import time
from datetime import datetime
from email.utils import parsedate_to_datetime
from concurrent.futures import ThreadPoolExecutor, as_completed
import requests
import pandas as pd
from bs4 import BeautifulSoup

# --- CẤU HÌNH ---
TARGET_ARTICLES = 100  # Số lượng bài viết cần thu thập thử nghiệm (100 - 500 bài)
MAX_WORKERS = 8        # Số luồng thu thập cố định

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/124.0.0.0"
}

CATEGORIES = [
    "thoi-su", "the-gioi", "kinh-doanh", "bat-dong-san", "khoa-hoc-cong-nghe",
    "giai-tri", "the-thao", "phap-luat", "giao-duc", "suc-khoe", "doi-song",
    "du-lich", "oto-xe-may", "tam-su", "y-kien", "thu-gian"
]

print(f"[+] Số bài viết mục tiêu: {TARGET_ARTICLES}")
print(f"[+] Số luồng xử lý      : {MAX_WORKERS}")
print(f"[+] Số chuyên mục quét  : {len(CATEGORIES)}")

[+] Số bài viết mục tiêu: 100
[+] Số luồng xử lý      : 8
[+] Số chuyên mục quét  : 16


## 2. Pha 1 — FETCH (Tải nội dung thô)
Gửi HTTP GET request với `User-Agent` hợp lệ và cơ chế xử lý lỗi/timeout để lấy mã nguồn trang.

In [2]:
def fetch(url):
    """Gửi request HTTP GET để lấy nội dung HTML của URL."""
    res = requests.get(url, headers=HEADERS, timeout=12)
    res.raise_for_status()
    res.encoding = "utf-8"
    return res.text

# Test thử tải trang RSS thời sự
sample_rss = fetch("https://vnexpress.net/rss/thoi-su.rss")
print(f"[+] Tải thành công RSS Thời sự: {len(sample_rss)} bytes")

[+] Tải thành công RSS Thời sự: 62859 bytes


## 3. Pha 2 — REFINE (Bóc tách & Chuẩn hóa Dữ liệu)
Trích xuất thông tin bài viết từ HTML:
- Chuẩn hóa ngày giờ về `YYYY-MM-DD HH:MM:SS`.
- Loại bỏ bài video / media (`og:type != article`).
- Khử nhãn breadcrumb giả `"Trang chủ"`.
- Bóc tách tác giả thông minh (bỏ ghi chú ảnh, video, biên tập ở cuối bài).

In [3]:
def normalize_datetime(raw):
    """Chuẩn hóa chuỗi ngày giờ từ web về định dạng YYYY-MM-DD HH:MM:SS."""
    if not raw:
        return None
    iso = re.search(r"\d{4}-\d{2}-\d{2}[ T]\d{2}:\d{2}", raw)
    if iso:
        return datetime.strptime(iso.group(0).replace("T", " "), "%Y-%m-%d %H:%M").strftime("%Y-%m-%d %H:%M:%S")

    dm = re.search(r"\d{1,2}/\d{1,2}/\d{4}", raw)
    tm = re.search(r"\d{1,2}:\d{2}", raw)
    if dm:
        dt = datetime.strptime(dm.group(0), "%d/%m/%Y")
        if tm:
            h, m = map(int, tm.group(0).split(":"))
            dt = dt.replace(hour=h, minute=m)
        return dt.strftime("%Y-%m-%d %H:%M:%S")
    return None


def refine_article(html, url, fallback_date=None):
    """Trích xuất các trường thông tin của bài viết từ mã nguồn HTML."""
    soup = BeautifulSoup(html, "html.parser")

    # Bỏ qua trang media/video không phải bài viết dạng text
    og = soup.find("meta", {"property": "og:type"})
    if og and og.get("content", "").lower() != "article":
        return None
    if not soup.find("h1"):
        return None

    # Thời gian xuất bản
    date_tag = soup.find("span", class_="date")
    raw_date = date_tag.text.strip() if date_tag else ""
    if not raw_date:
        m_date = soup.find("meta", {"name": "pubdate"})
        raw_date = m_date.get("content", "").strip() if m_date else ""
    published_at = normalize_datetime(raw_date) or fallback_date

    # Tiêu đề
    h1 = soup.find("h1", class_="title-detail") or soup.find("h1")
    title = h1.text.strip() if h1 else ""

    # Tóm tắt (description)
    desc_tag = soup.find("p", class_="description")
    description = desc_tag.text.strip() if desc_tag else ""
    if not description:
        m_desc = soup.find("meta", {"name": "description"}) or soup.find("meta", {"property": "og:description"})
        description = m_desc.get("content", "").strip() if m_desc else ""

    # Chuyên mục (Lọc bỏ nhãn giả 'Trang chủ')
    crumbs = [
        a.text.strip()
        for a in soup.select("ul.breadcrumb li a")
        if a.text.strip().lower() not in ["", "trang chủ"]
    ]
    category = crumbs[0] if crumbs else "Khác"
    subcategory = crumbs[1] if len(crumbs) > 1 else "Không có"

    # Nội dung và tác giả (Duyệt lùi 3 đoạn cuối, bỏ qua ghi chú ảnh/video)
    paragraphs = [
        p for p in soup.select(".fck_detail p.Normal, .fck_detail p, article p")
        if p.text.strip()
    ]
    author = "VnExpress"
    idx = len(paragraphs) - 1

    while idx >= 0 and (len(paragraphs) - 1 - idx) < 3:
        txt = paragraphs[idx].text.strip()
        strong = paragraphs[idx].find("strong")

        if txt.startswith(("*", "Ảnh:", "Video:")):
            paragraphs.pop(idx)
        elif strong and len(txt) < 50:
            author = strong.text.strip()
            paragraphs.pop(idx)
            break
        idx -= 1

    content = "\n".join(p.text.strip() for p in paragraphs)

    # Mã định danh bài viết
    meta_id = soup.find("meta", {"name": "tt_article_id"})
    if meta_id and meta_id.get("content"):
        aid = meta_id["content"]
    else:
        aid = url.split("?")[0].split("-")[-1].replace(".html", "")

    return {
        "article_id": aid,
        "title": title,
        "description": description,
        "content": content,
        "author": author,
        "published_at": published_at,
        "category": category,
        "subcategory": subcategory,
        "url": url,
    }

## 4. Pha 3 — VALIDATE (Kiểm định Chất lượng)
Đảm bảo bài báo có đầy đủ tiêu đề và nội dung tối thiểu từ 50 từ trở lên.

In [4]:
def validate_article(art):
    """Kiểm tra tính hợp lệ cơ bản của bài viết."""
    if not art:
        return False, "Không phải bài báo văn bản"

    if not art.get("title"):
        return False, "Thiếu tiêu đề"

    content = art.get("content", "")
    word_count = len(content.split())
    if word_count < 50:
        return False, f"Nội dung quá ngắn ({word_count} từ)"

    return True, "Hợp lệ"

## 5. Thu thập Liên kết (URLs) & Pipeline Xử lý
1. Quét nhanh từ 16 luồng RSS chuyên mục.
2. Nếu chưa đủ số lượng mục tiêu, quét tiếp phân trang chuyên mục (`p1 -> p20`).
3. Dùng `ThreadPoolExecutor` để tải và xử lý đồng thời nhiều bài viết.

In [5]:
def parse_pipeline(url, fallback_date):
    """Luồng xử lý một bài viết: tải -> bóc tách -> kiểm tra."""
    try:
        time.sleep(0.02)
        html = fetch(url)
        raw_article = refine_article(html, url, fallback_date)
        is_valid, reason = validate_article(raw_article)

        if is_valid:
            return raw_article, "OK"
        return None, reason
    except Exception as e:
        return None, f"Lỗi: {e}"


def collect_urls(target_count):
    """Thu thập danh sách URL bài viết từ RSS và phân trang chuyên mục."""
    print(f"[*] Thu thập liên kết bài viết (Mục tiêu: {target_count} bài)...")
    urls = {}

    # Quét RSS 16 chuyên mục
    for cat in CATEGORIES:
        try:
            xml = fetch(f"https://vnexpress.net/rss/{cat}.rss")
            soup = BeautifulSoup(xml, "xml")
            for item in soup.find_all("item"):
                link_tag = item.find("link")
                link = link_tag.text.strip() if link_tag else ""
                if not link.endswith(".html") or "/video/" in link:
                    continue

                pub_tag = item.find("pubDate")
                try:
                    dt = parsedate_to_datetime(pub_tag.text.strip())
                    urls[link.split("?")[0]] = (dt.timestamp(), dt.strftime("%Y-%m-%d %H:%M:%S"))
                except Exception:
                    urls[link.split("?")[0]] = (0.0, None)
        except Exception:
            pass

    # Quét trang chuyên mục nếu RSS chưa đủ số bài
    page = 1
    while len(urls) < target_count and page <= 20:
        added = 0
        for cat in CATEGORIES:
            try:
                html = fetch(f"https://vnexpress.net/{cat}-p{page}")
                soup = BeautifulSoup(html, "html.parser")
                for a in soup.select("article.item-news h3.title-news a[href]"):
                    href = a["href"].split("?")[0]
                    if href.startswith("/"):
                        href = "https://vnexpress.net" + href
                    if href.endswith(".html") and "/video/" not in href and href not in urls:
                        urls[href] = (0.0, None)
                        added += 1
                        if len(urls) >= target_count:
                            break
            except Exception:
                pass

            if len(urls) >= target_count:
                break

        if added == 0 and page > 3:
            break
        page += 1

    sorted_urls = sorted(urls.items(), key=lambda x: x[1][0], reverse=True)
    return sorted_urls[:target_count]

## 6. Thực thi Thu thập & Xuất Dữ liệu ra CSV
Chạy pipeline với đa luồng, thu thập bài viết thực tế và lưu file `vnexpress_data.csv`.

In [6]:
start_time = time.time()

# 1. Thu thập danh sách liên kết
links = collect_urls(TARGET_ARTICLES)
print(f"[+] Đã gom được {len(links)} link bài viết.")

# 2. Xử lý đa luồng qua ThreadPoolExecutor
articles = []
report = {}

print(f"[*] Bắt đầu tải và xử lý {len(links)} bài viết với {MAX_WORKERS} luồng...")
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {
        executor.submit(parse_pipeline, u, d[1]): u
        for u, d in links
    }
    for f in as_completed(futures):
        art, reason = f.result()
        report[reason] = report.get(reason, 0) + 1
        if art:
            articles.append(art)
            if len(articles) >= TARGET_ARTICLES:
                break

# 3. Tạo DataFrame & Lưu CSV
articles.sort(key=lambda x: str(x.get("published_at", "")), reverse=True)
df = pd.DataFrame(articles)
out_file = "vnexpress_data.csv"
df.to_csv(out_file, index=False, encoding="utf-8-sig")

elapsed = time.time() - start_time

print("\n" + "=" * 60)
print(" KẾT QUẢ THU THẬP ")
print("=" * 60)
print(f"[+] Số bài viết thu thập thành công : {len(df)} bài")
print(f"[+] File kết quả                   : {out_file}")
print(f"[+] Thời gian thực thi              : {elapsed:.1f} giây")
if not df.empty:
    print(f"[+] Dải thời gian                  : Từ [{df['published_at'].min()}] Đến [{df['published_at'].max()}]")

print("\n[+] Thống kê kiểm tra nội dung (Validation Report):")
for reason, count in report.items():
    print(f"  - {reason:<30}: {count:4d}")
print("=" * 60)

# Hiển thị mẫu 5 dòng đầu
df.head(5)

[*] Thu thập liên kết bài viết (Mục tiêu: 100 bài)...


[+] Đã gom được 100 link bài viết.
[*] Bắt đầu tải và xử lý 100 bài viết với 8 luồng...



 KẾT QUẢ THU THẬP 
[+] Số bài viết thu thập thành công : 69 bài
[+] File kết quả                   : vnexpress_data.csv
[+] Thời gian thực thi              : 17.7 giây
[+] Dải thời gian                  : Từ [2026-11-08 14:00:00] Đến [2026-11-09 08:00:00]

[+] Thống kê kiểm tra nội dung (Validation Report):
  - OK                            :   69
  - Lỗi: 429 Client Error: Too Many Requests for url: https://vnexpress.net/chuyen-tau-luon-cam-xuc-o-old-trafford-5142239.html:    1
  - Lỗi: 429 Client Error: Too Many Requests for url: https://vnexpress.net/barca-dap-tat-hy-vong-chiem-dinh-bang-cua-atletico-5142247.html:    1
  - Lỗi: 429 Client Error: Too Many Requests for url: https://vnexpress.net/nguoi-khmer-o-tp-hcm-mua-chan-trong-le-dang-y-5142165.html:    1
  - Lỗi: 429 Client Error: Too Many Requests for url: https://vnexpress.net/yangwang-altior-ke-thach-thuc-rolls-royce-tu-trung-quoc-5142145.html:    1
  - Lỗi: 429 Client Error: Too Many Requests for url: https://vnexpress.net/t

,article_id,title,description,content,author,published_at,category,subcategory,url
0,5141837,Căn hộ ven biển Đà Nẵng hưởng lợi từ dòng khác...,Nguồn khách du lịch tăng trưởng cùng nhu cầu l...,Nguồn khách du lịch tăng trưởng cùng nhu cầu l...,Song Anh,2026-11-09 08:00:00,Bất động sản,Thị trường,https://vnexpress.net/can-ho-ven-bien-da-nang-...
1,5142164,SSI: Chứng khoán có thể tạo đáy ngắn hạn trong...,VN-Index được dự báo tạo đáy ngắn hạn trong th...,VN-Index được dự báo tạo đáy ngắn hạn trong th...,Tất Đạt,2026-11-09 07:41:00,Kinh doanh,Chứng khoán,https://vnexpress.net/ssi-chung-khoan-co-the-t...
2,5142074,Đưa con trai 14 tuổi đi giao hàng để hiểu giá ...,Trung QuốcThấy con nói học không giỏi sau này ...,Trung QuốcThấy con nói học không giỏi sau này ...,Nhật Minh,2026-11-09 07:39:00,Đời sống,Bài học sống,https://vnexpress.net/dua-con-trai-14-tuoi-di-...
3,5142249,"Iran ra cảnh báo, yêu cầu Mỹ chấp nhận đề xuất...",Iran cảnh báo Mỹ phải chấp nhận đề xuất mới nh...,Iran cảnh báo Mỹ phải chấp nhận đề xuất mới nh...,Huyền Lê,2026-11-09 07:11:00,Thế giới,Không có,https://vnexpress.net/iran-ra-canh-bao-yeu-cau...
4,5142245,Gia Lai di dời khẩn cấp người dân ở núi Một do...,"Trước nguy cơ sạt lở do mưa lớn, phường Quy Nh...","Trước nguy cơ sạt lở do mưa lớn, phường Quy Nh...",Trần Hóa,2026-11-09 07:05:00,Thời sự,Không có,https://vnexpress.net/gia-lai-di-doi-khan-cap-...
